# 03 · Timeline reconstruction and robustness of the model evidence

This notebook documents two additions made when the project was redesigned (see `src/timeline.py`, `src/robustness.py`):
1. the CSV has no date, but its row order is chronological, so a month index can be reconstructed and *checked*;
2. the single 6-employee test set is replaced as the headline evidence by pooled grouped cross-validation over all 33 employees, employee-level bootstrap intervals and a time-ordered check.

Heavy computation is done by `python -m src.robustness`; this notebook reads its saved results from `models/metrics.json` so it runs in seconds.

In [1]:
import json, sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src import config
from src.data_loader import load_raw_data
from src.timeline import add_period, month_label
raw = load_raw_data()
with_period, check = add_period(raw)
check

{'runs_total': 38,
 'runs_with_month': 37,
 'consecutive_month_steps': 36,
 'steps_checked': 36,
 'one_month_per_run': True,
 'distinct_workload_values': 38,
 'rows_without_period': 3,
 'ok': True,
 'assumed_start': '2007-07',
 'note': 'Reconstructed from row order and the monthly workload figure; the file has no date column. The start month is taken from the public dataset documentation.'}

## 1 · Is row order chronological?
Evidence: the monthly company figure *Work load Average/day* forms contiguous runs, one per calendar month, and the month of successive runs advances by exactly one month. A shuffled copy must fail the same check:

In [2]:
shuffled = raw.sample(frac=1, random_state=1).reset_index(drop=True)
print('real order  ->', add_period(raw)[1]['ok'])
print('shuffled    ->', add_period(shuffled)[1]['ok'])
print(month_label(1), '...', month_label(37))

real order  -> True
shuffled    -> False
2007-07 ... 2010-07


In [3]:
events = with_period.dropna(subset=['period']).groupby('period').size()
events.describe().round(1)

count    37.0
mean     19.9
std       5.8
min      11.0
25%      16.0
50%      19.0
75%      22.0
max      36.0
dtype: float64

The start month (July 2007) is **not** in the file; it comes from the public dataset documentation, so month labels are an assumption that the app labels as such.

## 2 · Pooled grouped cross-validation (all 33 employees held out once per repeat)

In [4]:
v2 = json.loads(config.METRICS_PATH.read_text())['evaluation_v2']
pd.DataFrame(v2['pooled_grouped_cv']).T[['pr_auc','roc_auc','brier','recall','precision','f1']].round(3)

,pr_auc,roc_auc,brier,recall,precision,f1
Reason-only baseline (LR),0.726809,0.854661,0.133179,0.904059,0.731334,0.808568
Logistic Regression,0.728529,0.852665,0.137403,0.899631,0.726048,0.803562
Random Forest,0.743797,0.858047,0.136678,0.904428,0.719221,0.801246
SVM (RBF kernel),0.753451,0.856728,0.13376,0.872325,0.728547,0.793696
Smoothed reason lookup,0.731509,0.85718,0.137032,0.893358,0.732081,0.804663
Prevalence baseline (no skill),0.388809,0.5,0.237637,1.0,0.388809,0.559917


## 3 · Is the deployed Random Forest better than the baselines?
Paired PR-AUC differences with 95% employee-level bootstrap intervals; an interval containing 0 means the difference cannot be told apart from noise.

In [5]:
pd.DataFrame({k: {'median': v['median'], 'ci_low': v['ci95'][0], 'ci_high': v['ci95'][1]} for k, v in v2['paired_pr_auc_differences'].items()}).T.round(3)

,median,ci_low,ci_high
Random Forest minus Reason-only baseline (LR),0.022,-0.006,0.050
SVM (RBF kernel) minus Reason-only baseline (LR),0.040,-0.002,0.088
Logistic Regression minus Reason-only baseline (LR),0.006,-0.023,0.032
Random Forest minus Smoothed reason lookup,0.019,-0.012,0.047


## 4 · Time-ordered check (earlier months → later months)

In [6]:
t = v2['temporal_check']
print({k: t[k] for k in ['train_months','test_months','train_rows','test_rows','train_prevalence','test_prevalence','test_employees','test_employees_also_in_train']})
pd.DataFrame(t['models']).T.round(3)

{'train_months': 25, 'test_months': 12, 'train_rows': 448, 'test_rows': 249, 'train_prevalence': 0.421875, 'test_prevalence': 0.3293172690763052, 'test_employees': 26, 'test_employees_also_in_train': 22}


,pr_auc,roc_auc,recall,precision,f1
Reason-only baseline (LR),0.670,0.823,0.878,0.626,0.731
Logistic Regression,0.667,0.824,0.878,0.600,0.713
Random Forest,0.687,0.839,0.890,0.584,0.705
SVM (RBF kernel),0.669,0.840,0.878,0.590,0.706


## Conclusion
The reason code carries almost all of the usable signal. The Random Forest, SVM and the one-column baselines cannot be separated statistically with 33 employees, so the deployed model is kept because nothing justifies replacing it, not because it is clearly better.